# Minimum Set Cover: giới thiệu, chạy thuật toán và đánh giá

Notebook này sinh dữ liệu, chạy sáu phương pháp và so sánh số tập cùng thời gian:
Greedy, Greedy + cleanup, Randomized Greedy, Brute Force, Dynamic Programming
và Branch and Bound. Greedy + cleanup là đối chứng để tách tác dụng loại tập thừa
khỏi tác dụng của các lần dựng nghiệm ngẫu nhiên.

Chạy từ thư mục gốc repository hoặc `notebooks/`. Cài môi trường đang dùng:

```bash
python -m pip install -e ".[notebook]"
jupyter notebook notebooks/minimum_cover_set_evaluation.ipynb
```

Đây là minh họa với một seed và một lần đo mỗi phương pháp. Các thí nghiệm nhiều
seed sử dụng các cấu hình trong `experiments/`; kết quả mới được lưu vào `results/`.


## 1. Chuẩn bị môi trường

Các import dưới đây sử dụng trực tiếp source code trong thư mục `src/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/min_cover_set/model.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/.")
sys.path.insert(0, str(REPO_ROOT / "src"))

from data_generator import generate_instance
from min_cover_set import MinimumCoverSet
from min_cover_set.validation import validate_cover


## 2. Sinh dữ liệu

Giữ `SEED` cố định khi so sánh solver. `SOLVER_SEED` chỉ điều khiển Randomized Greedy;
`RESTARTS` là số lần dựng nghiệm ngẫu nhiên, ngoài nghiệm nền Greedy + cleanup;
`RCL_SIZE` là số ứng viên có độ phủ mới tốt nhất được phép chọn ngẫu nhiên.

Bộ sinh luôn gán mỗi phần tử cho ít nhất một tập. `DENSITY` là xác suất thêm phủ
sau bước gán này, không phải mật độ thực đo chính xác.

Notebook gọi cả ba thuật toán chính xác nên chỉ dùng kích thước có thể giải hết.
Có thể thử `NUM_ELEMENTS = 0` để kiểm tra trường hợp universe rỗng.


In [ ]:
NUM_ELEMENTS = 20
NUM_SUBSETS = 12
DENSITY = 0.30
SEED = 42
SOLVER_SEED = 0
RESTARTS = 30
RCL_SIZE = 3

instance = generate_instance(
    num_elements=NUM_ELEMENTS,
    num_subsets=NUM_SUBSETS,
    density=DENSITY,
    seed=SEED,
)

subset_sizes = {
    str(name): len(elements)
    for name, elements in instance.subsets.items()
}

print(f"Universe: {len(instance.universe)} elements")
print(f"Subsets: {len(instance.subsets)}")
print(f"Generator density: {DENSITY}, generator seed: {SEED}")
print(f"Solver seed: {SOLVER_SEED}, restarts: {RESTARTS}, RCL: {RCL_SIZE}")
pd.DataFrame(
    sorted(subset_sizes.items()),
    columns=["subset", "number_of_elements"],
)


## 3. Chạy lần lượt các thuật toán

Brute Force cung cấp nghiệm tham chiếu tối ưu. Mỗi lần đo bao gồm toàn bộ lời gọi
solver, gồm nghiệm nền và tất cả restarts/cleanup đối với Randomized Greedy.
Kiểm tra nghiệm độc lập thực hiện sau phép đo. Tất cả phương pháp cùng nhận một
instance và trả về tuple tên tập theo giao diện chung của nhóm.


In [ ]:
cover = MinimumCoverSet(instance)
algorithms = {
    "Greedy": cover.greedy,
    "Greedy + cleanup": cover.greedy_pruned,
    "Randomized Greedy": lambda: cover.randomized_greedy(
        seed=SOLVER_SEED, restarts=RESTARTS, rcl_size=RCL_SIZE
    ),
    "Brute Force": cover.brute_force,
    "Dynamic Programming": cover.dynamic_programming,
    "Branch and Bound": cover.branch_and_bound,
}

results = []
optimal_solution = None
optimal_size = None

for name, algorithm in algorithms.items():
    start = time.perf_counter()
    selected = algorithm()
    elapsed_ms = (time.perf_counter() - start) * 1000
    validate_cover(instance, selected)

    if name == "Brute Force":
        optimal_solution = selected
        optimal_size = len(selected)

    results.append(
        {
            "algorithm": name,
            "selected": len(selected),
            "runtime_ms": elapsed_ms,
            "solution": list(selected),
            "valid": True,
        }
    )

results_df = pd.DataFrame(results)
results_df["optimal"] = results_df["selected"] == optimal_size
results_df["accuracy"] = results_df["selected"].map(
    lambda size: optimal_size / size if size else 1.0
)
results_df["optimality_gap_pct"] = (
    100 * (results_df["selected"] - optimal_size) / optimal_size
    if optimal_size else 0.0
)
results_df[["algorithm", "selected", "optimal", "accuracy", "optimality_gap_pct",
            "valid", "runtime_ms"]]


## 4. Kết quả và so sánh

- `selected`: số tập được chọn; `solution` chứa các tên tập cụ thể.
- `optimal`: có bằng kích thước nghiệm Brute Force hay không (boolean của notebook).
- `accuracy`: `optimal_size / selected`; quy ước bằng 1 khi universe rỗng.
- `optimality_gap_pct`: phần trăm số tập vượt quá nghiệm tối ưu; bằng 0 cho bài rỗng.
- `valid`: nghiệm đã qua kiểm tra tên tập, không lặp và phủ đủ universe.
- `runtime_ms`: thời gian gọi solver, tính bằng milliseconds.

Trường `optimal` trong `src/evaluate.py` vẫn là **số tập tối ưu** theo API cũ;
notebook giữ kiểu boolean mà nhóm đã dùng. Thời gian một lần chạy có thể dao động,
không đủ để kết luận tốc độ trung bình hay độ ổn định của heuristic.


In [ ]:
display(
    results_df.assign(
        accuracy=results_df["accuracy"].round(2),
        runtime_ms=results_df["runtime_ms"].round(3),
    )
)

fastest = results_df.loc[results_df["runtime_ms"].idxmin()]
print(
    f"Fastest algorithm: {fastest['algorithm']} "
    f"({fastest['runtime_ms']:.3f} ms)"
)
print(f"Optimal solution size: {optimal_size}")

## 5. Trực quan hóa

Biểu đồ trái so sánh số tập; đường nét đứt là nghiệm tối ưu từ Brute Force.
Biểu đồ phải hiển thị thời gian của lần chạy hiện tại, gồm toàn bộ 30 restarts
nếu giữ tham số mặc định. Thang đo tuyến tính có thể làm các solver rất nhanh
trông gần bằng 0; xem bảng phía trên để đọc giá trị chính xác.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

results_df.plot.bar(
    x="algorithm", y="selected", ax=axes[0], color="#4C78A8", legend=False,
    title="Solution size",
)
axes[0].axhline(optimal_size, color="#444444", linestyle="--", label="Exact optimum")
axes[0].set_ylabel("Number of selected subsets")
axes[0].legend()

results_df.plot.bar(
    x="algorithm", y="runtime_ms", ax=axes[1], color="#F58518", legend=False,
    title="Runtime for this run",
)
axes[1].set_ylabel("Runtime (ms)")
for ax in axes:
    ax.set_xlabel("")
    ax.set_ylim(bottom=0)
    ax.tick_params(axis="x", rotation=35)
    plt.setp(ax.get_xticklabels(), ha="right")

fig.suptitle("Minimum Set Cover evaluation", fontsize=14)
fig.tight_layout()
plt.show()
